# Workshop 2 (Your version)

This notebook downloads an **art-related image dataset** from Panoramas of Cinema search results and saves:
- images to `dataset/poc_arcade/`
- `metadata.csv` with source URLs + download status

> Why this version? Your current Python is **3.14**, and packages like **numpy/pandas/pillow/torch/faiss** may not install yet. This notebook avoids them so it **runs reliably**.


In [8]:
import sys, subprocess, textwrap, os

# Install only lightweight deps that work on Python 3.14
# If you already installed them, this is quick.
pkgs = ["selenium", "requests", "tqdm"]
print("Python:", sys.version)
print("Installing:", pkgs)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("Done.")


Python: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
Installing: ['selenium', 'requests', 'tqdm']
Done.


In [9]:
import os, re, csv, time, hashlib
from urllib.parse import urljoin, urlparse
import requests
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

# ----------------------------
# CONFIG (edit these)
# ----------------------------
START_URL = "https://panoramasofcinema.ch/search?q=places=arcade,0.010,1.000+faces=no_faces,small_faces"
OUT_DIR = os.path.join("dataset", "poc_arcade")
META_CSV = os.path.join(OUT_DIR, "metadata.csv")

MAX_IMAGES = 120          # stop after collecting this many image URLs
SCROLL_STEPS = 18         # how many scrolls to trigger lazy-loading
SCROLL_PAUSE = 0.7        # seconds between scrolls
HEADLESS = False          # first run: False so you can see what's happening
REQUEST_TIMEOUT = 25
SLEEP_BETWEEN_DOWNLOADS = 0.15

os.makedirs(OUT_DIR, exist_ok=True)

def safe_filename(s: str, max_len: int = 120) -> str:
    s = (s or "").strip()
    s = re.sub(r'[\\/:*?"<>|]+', "", s)
    s = re.sub(r"\s+", " ", s).strip()
    if not s:
        s = "image"
    return s[:max_len].strip()

def pick_ext_from_url(url: str) -> str:
    path = urlparse(url).path.lower()
    for ext in (".jpg",".jpeg",".png",".webp",".gif",".bmp"):
        if path.endswith(ext):
            return ext
    return ".jpg"  # fallback

def download_file(url: str, out_path: str):
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, stream=True, timeout=REQUEST_TIMEOUT, headers=headers)
    r.raise_for_status()
    ctype = r.headers.get("Content-Type","").lower()
    # accept images only
    if "image" not in ctype:
        raise ValueError(f"Not an image content-type: {ctype}")
    with open(out_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)


In [10]:
def build_driver(headless: bool):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,900")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(8)
    return driver

def collect_image_urls(start_url: str, max_images: int = 120, scroll_steps: int = 18):
    driver = build_driver(HEADLESS)
    urls = set()
    try:
        driver.get(start_url)
        time.sleep(1.2)

        for step in range(scroll_steps):
            # Collect <img> sources (src + common lazy-load attrs)
            imgs = driver.find_elements(By.CSS_SELECTOR, "img")
            for im in imgs:
                for attr in ("src","data-src","data-lazy","data-original","srcset"):
                    try:
                        v = im.get_attribute(attr)
                    except Exception:
                        v = None
                    if not v:
                        continue
                    # srcset may contain multiple URLs
                    parts = [p.strip().split(" ")[0] for p in v.split(",")]
                    for u in parts:
                        if u and u.startswith("http"):
                            urls.add(u)

            # Collect background-image urls
            els = driver.find_elements(By.CSS_SELECTOR, "*")
            for el in els[:400]:  # limit to avoid being too slow
                try:
                    bg = el.value_of_css_property("background-image")
                except Exception:
                    bg = ""
                if bg and "url(" in bg:
                    m = re.findall(r'url\(["\']?(.*?)["\']?\)', bg)
                    for u in m:
                        if u.startswith("http"):
                            urls.add(u)

            print(f"[collect] step {step+1}/{scroll_steps}, urls={len(urls)}")
            if len(urls) >= max_images:
                break

            # Scroll down to trigger lazy load
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(SCROLL_PAUSE)

    finally:
        driver.quit()

    # Filter out obvious non-image URLs
    filtered = []
    for u in urls:
        if any(x in u.lower() for x in (".jpg",".jpeg",".png",".webp",".gif",".bmp")):
            filtered.append(u)
    # If site uses image endpoints without extensions, keep some extras
    if len(filtered) < 10:
        filtered = list(urls)

    filtered = sorted(filtered)[:max_images]
    return filtered

image_urls = collect_image_urls(START_URL, max_images=MAX_IMAGES, scroll_steps=SCROLL_STEPS)
print("Collected image URLs:", len(image_urls))
print("First 5:")
for u in image_urls[:5]:
    print(" -", u)


[collect] step 1/18, urls=200
Collected image URLs: 120
First 5:
 - https://panoramas-of-cinema.s3.eu-central-1.amazonaws.com/frames_db/1900_part1/by_scene/frame-220880.jpg
 - https://panoramas-of-cinema.s3.eu-central-1.amazonaws.com/frames_db/8_and_a_half/by_scene/frame-025912.jpg
 - https://panoramas-of-cinema.s3.eu-central-1.amazonaws.com/frames_db/96/by_scene/frame-029448.jpg
 - https://panoramas-of-cinema.s3.eu-central-1.amazonaws.com/frames_db/96/by_scene/frame-052488.jpg
 - https://panoramas-of-cinema.s3.eu-central-1.amazonaws.com/frames_db/96/by_scene/frame-218424.jpg


In [11]:
rows = []
downloaded = 0
failed = 0

for idx, url in enumerate(tqdm(image_urls, desc="Downloading images")):
    # stable filename based on url hash
    h = hashlib.md5(url.encode("utf-8")).hexdigest()[:10]
    ext = pick_ext_from_url(url)
    fname = f"poc_arcade_{idx:04d}_{h}{ext}"
    out_path = os.path.join(OUT_DIR, fname)

    status = "ok"
    err = ""
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1500:
        downloaded += 1
    else:
        try:
            download_file(url, out_path)
            # basic size check
            if os.path.getsize(out_path) < 1500:
                raise ValueError("Downloaded file too small")
            downloaded += 1
            time.sleep(SLEEP_BETWEEN_DOWNLOADS)
        except Exception as e:
            status = "fail"
            err = repr(e)
            failed += 1
            # remove bad file if created
            try:
                if os.path.exists(out_path):
                    os.remove(out_path)
            except Exception:
                pass

    rows.append({
        "index": idx,
        "url": url,
        "file": out_path,
        "status": status,
        "error": err
    })

# Save metadata.csv
with open(META_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["index","url","file","status","error"])
    w.writeheader()
    w.writerows(rows)

print("[DONE] Downloaded:", downloaded, "Failed:", failed)
print("[DONE] Dataset folder:", os.path.abspath(OUT_DIR))
print("[DONE] Metadata CSV:", os.path.abspath(META_CSV))


[DONE] Downloaded: 0 Failed: 120
[DONE] Dataset folder: C:\Users\16598\dataset\poc_arcade
[DONE] Metadata CSV: C:\Users\16598\dataset\poc_arcade\metadata.csv


In [12]:
import glob
imgs = []
for ext in ("*.jpg","*.jpeg","*.png","*.webp","*.gif","*.bmp"):
    imgs += glob.glob(os.path.join(OUT_DIR, ext))
print("Local images found:", len(imgs))
print("Example paths:", imgs[:3])


Local images found: 0
Example paths: []


In [13]:
import os

print("Notebook working dir:", os.getcwd())
print("Images folder:", os.path.abspath("dataset/poc_arcade"))


Notebook working dir: C:\Users\16598
Images folder: C:\Users\16598\dataset\poc_arcade


In [14]:
# ====== PoC (panoramasofcinema.ch) downloader: robust version ======
# 作用：从搜索页抓结果 -> 进入每个作品详情页 -> 提取真实图片(og:image优先) -> 下载到本地

import os, re, csv, time, hashlib
from urllib.parse import urljoin
import requests

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

SEARCH_URL = "https://panoramasofcinema.ch/search?q=places=arcade,0.010,1.000+faces=no_faces,small_faces"

OUT_DIR = r"dataset\poc_arcade"
META_CSV = os.path.join(OUT_DIR, "metadata.csv")
MAX_ITEMS = 120          # 你想下多少个结果
HEADLESS = False         # 第一次建议 False
SCROLL_TIMES = 20        # 滚动次数：越大抓到越多结果

os.makedirs(OUT_DIR, exist_ok=True)

def build_driver(headless=False):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1400,900")
    driver = webdriver.Chrome(options=opts)
    driver.implicitly_wait(10)
    return driver

def safe_filename(s, max_len=80):
    s = (s or "").strip()
    s = re.sub(r'[\\/:*?"<>|]+', "", s)
    s = re.sub(r"\s+", " ", s).strip()
    if not s:
        s = "item"
    return s[:max_len]

def get_requests_session_from_selenium(driver):
    """把 Selenium 的 cookie 带到 requests 里，解决很多站点的防盗链/权限问题"""
    sess = requests.Session()
    for c in driver.get_cookies():
        sess.cookies.set(c["name"], c["value"])
    # 伪装浏览器
    sess.headers.update({
        "User-Agent": "Mozilla/5.0",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    })
    return sess

def collect_result_links(driver, max_items=120):
    """从搜索页收集作品详情页链接（优先收集 a[href*='/pano' or '/panorama' or '/view']，不行就收集所有结果卡片链接）"""
    driver.get(SEARCH_URL)
    time.sleep(1)

    links = []
    for _ in range(SCROLL_TIMES):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(0.6)

        a_tags = driver.find_elements(By.CSS_SELECTOR, "a[href]")
        cand = []
        for a in a_tags:
            href = a.get_attribute("href") or ""
            # 这里放宽一些，避免站点结构变动导致抓不到
            if "panoramasofcinema.ch" in href and ("search" not in href):
                cand.append(href)

        # 去重保持顺序
        for u in cand:
            if u not in links:
                links.append(u)
        if len(links) >= max_items:
            break

    return links[:max_items]

def extract_image_from_detail(driver):
    """在详情页提取真实图片链接：优先 og:image -> 次选最大 img"""
    # 1) og:image
    og = driver.find_elements(By.CSS_SELECTOR, "meta[property='og:image'], meta[name='og:image']")
    if og:
        content = og[0].get_attribute("content")
        if content and content.startswith("http"):
            return content

    # 2) 常规 img（挑一个分辨率最大的）
    imgs = driver.find_elements(By.CSS_SELECTOR, "img")
    best = ""
    best_area = -1
    for im in imgs:
        src = im.get_attribute("src") or ""
        if not src:
            continue
        # 过滤图标/头像之类
        if any(x in src.lower() for x in ["logo", "icon", "avatar", "sprite"]):
            continue
        w = im.get_attribute("naturalWidth") or im.get_attribute("width") or "0"
        h = im.get_attribute("naturalHeight") or im.get_attribute("height") or "0"
        try:
            area = int(w) * int(h)
        except:
            area = 0
        if area > best_area and src.startswith("http"):
            best_area = area
            best = src
    return best

def download_image(sess, img_url, referer, out_path):
    # 很多站点需要 referer
    headers = {"Referer": referer}
    r = sess.get(img_url, headers=headers, stream=True, timeout=30)
    r.raise_for_status()
    ctype = r.headers.get("Content-Type","").lower()
    if "image" not in ctype:
        raise RuntimeError(f"not image content-type: {ctype}")

    with open(out_path, "wb") as f:
        for chunk in r.iter_content(8192):
            if chunk:
                f.write(chunk)
    if os.path.getsize(out_path) < 1500:
        raise RuntimeError("downloaded file too small")

# ----------------- RUN -----------------
driver = build_driver(headless=HEADLESS)
rows = []
downloaded = 0
failed = 0

try:
    # 1) 收集详情页链接
    detail_links = collect_result_links(driver, max_items=MAX_ITEMS)
    print("[INFO] detail links collected:", len(detail_links))
    if not detail_links:
        print("[HINT] 没抓到链接：网站结构可能和我假设不一致。把页面 HTML 结构发我我会改 selector。")

    # 2) 用 selenium cookie 建 requests session
    sess = get_requests_session_from_selenium(driver)

    # 3) 逐个进入详情页 -> 抓 image -> 下载
    for idx, link in enumerate(detail_links, start=1):
        try:
            driver.get(link)
            time.sleep(0.6)

            img_url = extract_image_from_detail(driver)
            if not img_url:
                raise RuntimeError("no image url found on detail page")

            # 绝对化
            img_url = urljoin(link, img_url)

            # 文件名：idx + hash 防重名
            h = hashlib.md5(img_url.encode("utf-8")).hexdigest()[:10]
            fn = f"{idx:03d}_{h}.jpg"
            out_path = os.path.join(OUT_DIR, fn)

            download_image(sess, img_url, referer=link, out_path=out_path)

            rows.append({"index": idx, "detail_url": link, "image_url": img_url, "file": fn, "status": "ok", "error": ""})
            downloaded += 1

        except Exception as e:
            rows.append({"index": idx, "detail_url": link, "image_url": "", "file": "", "status": "failed", "error": repr(e)})
            failed += 1

        if idx % 20 == 0:
            print(f"[PROGRESS] {idx}/{len(detail_links)} downloaded={downloaded} failed={failed}")

finally:
    try:
        driver.quit()
    except:
        pass

# 4) 保存 metadata（即使失败也会保存原因）
with open(META_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["index","detail_url","image_url","file","status","error"])
    w.writeheader()
    w.writerows(rows)

print("[DONE] Downloaded:", downloaded, "Failed:", failed)
print("[DONE] Dataset folder:", os.path.abspath(OUT_DIR))
print("[DONE] Metadata CSV:", os.path.abspath(META_CSV))


[INFO] detail links collected: 4
[DONE] Downloaded: 0 Failed: 4
[DONE] Dataset folder: C:\Users\16598\dataset\poc_arcade
[DONE] Metadata CSV: C:\Users\16598\dataset\poc_arcade\metadata.csv


In [ ]:
import os, re, csv, time, hashlib
from urllib.parse import urljoin, urlparse
import requests

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

SEARCH_URL = "https://panoramasofcinema.ch/search?q=places=arcade,0.010,1.000+faces=no_faces,small_faces"

OUT_DIR = r"dataset\poc_arcade"
META_CSV = os.path.join(OUT_DIR, "metadata.csv")
MAX_ITEMS = 120
HEADLESS = False

os.makedirs(OUT_DIR, exist_ok=True)

def build_driver(headless=False):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1400,900")
    driver = webdriver.Chrome(options=opts)
    driver.implicitly_wait(10)
    return driver

def try_click_cookie(driver):
    # 尝试点击常见的 cookie/consent 按钮（不保证，但经常有用）
    candidates = [
        "button", "a", "[role='button']", "input[type='button']", "input[type='submit']"
    ]
    keywords = ["accept", "agree", "ok", "got it", "allow", "i understand"]
    for sel in candidates:
        els = driver.find_elements(By.CSS_SELECTOR, sel)
        for e in els[:80]:
            txt = (e.text or "").strip().lower()
            if any(k in txt for k in keywords):
                try:
                    e.click()
                    time.sleep(0.8)
                    return True
                except:
                    pass
    return False

def looks_like_detail_link(u: str) -> bool:
    # 过滤掉资源文件、锚点、search 自己等
    if not u:
        return False
    u = u.strip()
    if u.startswith("javascript:") or u.startswith("mailto:"):
        return False
    low = u.lower()
    if "search?" in low or "/search" in low:
        return False
    if any(low.endswith(ext) for ext in [".css",".js",".svg",".png",".jpg",".jpeg",".webp",".gif",".ico",".woff",".woff2",".ttf"]):
        return False
    # 必须是本域名
    return "panoramasofcinema.ch" in low

def normalize(u: str) -> str:
    # 去掉 hash，减少重复
    try:
        p = urlparse(u)
        return p._replace(fragment="").geturl()
    except:
        return u

def collect_detail_links(driver, max_items=120, max_rounds=30):
    driver.get(SEARCH_URL)
    time.sleep(1.2)
    try_click_cookie(driver)

    links = []
    seen = set()

    def harvest():
        a_tags = driver.find_elements(By.CSS_SELECTOR, "a[href]")
        for a in a_tags:
            href = a.get_attribute("href") or ""
            href = urljoin(SEARCH_URL, href)
            href = normalize(href)
            if looks_like_detail_link(href) and href not in seen:
                seen.add(href)
                links.append(href)

    # 同时做：滚动 + 找“load more”按钮
    for r in range(max_rounds):
        harvest()
        if len(links) >= max_items:
            break

        # 1) 尝试点击 “load more / more / show more”
        clicked = False
        for kw in ["load more", "more", "show more", "weiter", "plus", "more results"]:
            btns = driver.find_elements(By.XPATH, f"//button[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'), '{kw}')]")
            if btns:
                try:
                    btns[0].click()
                    clicked = True
                    time.sleep(1.0)
                    break
                except:
                    pass

        # 2) 如果没按钮，就滚动
        if not clicked:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1.0)

        # 每轮都试着点一次 cookie
        try_click_cookie(driver)

    return links[:max_items]

def get_requests_session_from_selenium(driver):
    sess = requests.Session()
    for c in driver.get_cookies():
        sess.cookies.set(c["name"], c["value"])
    sess.headers.update({
        "User-Agent": "Mozilla/5.0",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    })
    return sess

def extract_image_url(driver):
    # 优先 og:image
    og = driver.find_elements(By.CSS_SELECTOR, "meta[property='og:image'], meta[name='og:image']")
    if og:
        u = og[0].get_attribute("content")
        if u:
            return u

    # 次选：页面中所有 img 的 src/srcset
    imgs = driver.find_elements(By.CSS_SELECTOR, "img")
    cand = []
    for im in imgs:
        src = (im.get_attribute("src") or "").strip()
        srcset = (im.get_attribute("srcset") or "").strip()
        if src and src.startswith("http"):
            cand.append(src)
        if srcset:
            # srcset 形如: url1 320w, url2 640w
            parts = [p.strip().split(" ")[0] for p in srcset.split(",") if p.strip()]
            for p in parts:
                if p.startswith("http"):
                    cand.append(p)
    # 去掉明显的 logo/icon
    cand2 = [u for u in cand if not any(x in u.lower() for x in ["logo","icon","avatar","sprite"])]
    return cand2[-1] if cand2 else ""

def download_image(sess, img_url, referer, out_path):
    r = sess.get(img_url, headers={"Referer": referer}, stream=True, timeout=30)
    r.raise_for_status()
    ctype = (r.headers.get("Content-Type","") or "").lower()
    if "image" not in ctype:
        raise RuntimeError(f"not image content-type: {ctype}")
    with open(out_path, "wb") as f:
        for chunk in r.iter_content(8192):
            if chunk:
                f.write(chunk)
    if os.path.getsize(out_path) < 1500:
        raise RuntimeError("downloaded file too small")

# ---------------- RUN ----------------
driver = build_driver(headless=HEADLESS)
rows = []
downloaded = 0
failed = 0

try:
    detail_links = collect_detail_links(driver, max_items=MAX_ITEMS, max_rounds=40)
    print("[INFO] detail links collected:", len(detail_links))
    print("[INFO] first 20 links:")
    for u in detail_links[:20]:
        print(" -", u)

    sess = get_requests_session_from_selenium(driver)

    for idx, link in enumerate(detail_links, start=1):
        try:
            driver.get(link)
            time.sleep(0.8)

            img_url = extract_image_url(driver)
            if not img_url:
                raise RuntimeError("no image url found on detail page")

            img_url = urljoin(link, img_url)

            h = hashlib.md5(img_url.encode("utf-8")).hexdigest()[:10]
            fn = f"{idx:03d}_{h}.jpg"
            out_path = os.path.join(OUT_DIR, fn)

            download_image(sess, img_url, referer=link, out_path=out_path)

            rows.append({"index": idx, "detail_url": link, "image_url": img_url, "file": fn, "status": "ok", "error": ""})
            downloaded += 1

        except Exception as e:
            rows.append({"index": idx, "detail_url": link, "image_url": "", "file": "", "status": "failed", "error": repr(e)})
            failed += 1

        if idx % 10 == 0:
            print(f"[PROGRESS] {idx}/{len(detail_links)} downloaded={downloaded} failed={failed}")

finally:
    try:
        driver.quit()
    except:
        pass

with open(META_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["index","detail_url","image_url","file","status","error"])
    w.writeheader()
    w.writerows(rows)

print("[DONE] Downloaded:", downloaded, "Failed:", failed)
print("[DONE] Dataset folder:", os.path.abspath(OUT_DIR))
print("[DONE] Metadata CSV:", os.path.abspath(META_CSV))
